# Stage 0: smoke test

Checks, on a Colab GPU, that the selector model (Qwen3.5-0.8B-Base + element adapter + pointer and action heads + LoRA) loads, trains in each number format, can learn a synthetic pointing task, and saves and resumes correctly.

Runtime: **T4 GPU**. Run all cells; each run ends with a JSON report.

In [19]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [20]:
%cd /content
!rm -rf bettercomputer && git clone -q --depth 1 https://github.com/oxums/bettercomputer
!pip install -q -e ./bettercomputer
!cd bettercomputer && git log --oneline -1

/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for bettercomputer (pyproject.toml) ... done
cdc47d6 (grafted, HEAD -> main, origin/main, origin/HEAD) Add text-as-tokens element mode


## Run

The first run on a fresh machine spends about 5 minutes per number format compiling the recurrent-layer kernels. Later runs reuse them.

In [21]:
!cd /content/bettercomputer && python -m bettercomputer.smoke --steps 500 --text-mode tokens --batch-size 8 --eval-batches 32 --formats fp16 --out /content/smoke_report.json

{'device': 'Tesla T4', 'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'recurrent_kernel': 'flash-linear-attention'}
Loading weights: 100% 320/320 [00:01<00:00, 274.63it/s]
loaded: {'total': 765382857, 'trainable': 12989833, 'lora_layers': 150}
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
  fp16 pass 1: 9.01s loss 4.902
  fp16 pass 2: 0.77s loss 4.902
  fp16 pass 3: 0.77s loss 4.902
  fp16 pass 4: 0.77s loss 4.902
  fp16 pass 5: 0.77s loss 4.902
  fp16 pass 6: 0.77s loss 4.902
format fp16 {'ok': True, 'loss': 4.9023, 'grads_finite': True, 's_per_step': 0.7677, 'tokens_per_s': 344, 'peak_mem_gb': 7.08}
before training: {'pointer_acc': 0.140625, 'elem_acc': 0.0, 'action_acc': 0.3125, 'none_acc': 0.8780487804878049, 'samples': 256}
{'step': 20, 'loss': 2.5723, 'tokens_per_s': 313, 's_per_step': 0.961}
{'step': 40, 'los